# Titanic 생존 예측 학습 및 검증

위에서 아래로 실행하면 데이터와 학습 코드를 검사한 뒤 모델을 학습함. `test.csv`에는 정답이 없어 훈련과 검증에 사용하지 않음.

In [ ]:
import math
import sys
import unittest
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import wandb

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "_03_homeworks/homework_2/titanic_dataset.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("프로젝트 디렉터리 안에서 노트북을 실행해야 함")
sys.path.insert(0, str(project_root))

from _03_homeworks.homework_2.titanic_dataset import get_preprocessed_dataset

## 데이터 로딩과 모델 학습 코드

In [ ]:
def get_data(run):
  train_dataset, validation_dataset, _ = get_preprocessed_dataset()
  train_data_loader = DataLoader(
    dataset=train_dataset, batch_size=run.config.batch_size, shuffle=True
  )
  validation_data_loader = DataLoader(
    dataset=validation_dataset, batch_size=run.config.batch_size
  )
  return train_data_loader, validation_data_loader


class MyModel(nn.Module):
  def __init__(self, n_input, n_output, run):
    super().__init__()
    hidden_units = run.config.n_hidden_unit_list
    self.model = nn.Sequential(
      nn.Linear(n_input, hidden_units[0]),
      nn.ReLU(),
      nn.Linear(hidden_units[0], hidden_units[1]),
      nn.ReLU(),
      nn.Linear(hidden_units[1], n_output),
    )

  def forward(self, x):
    return self.model(x)


def get_model_and_optimizer(run, n_input):
  model = MyModel(n_input=n_input, n_output=2, run=run)
  optimizer = optim.Adam(model.parameters(), lr=run.config.learning_rate)
  return model, optimizer


def run_epoch(model, data_loader, loss_fn, optimizer=None):
  is_training = optimizer is not None
  model.train(is_training)
  total_loss = 0.0
  total_correct = 0
  total_samples = 0

  with torch.set_grad_enabled(is_training):
    for batch in data_loader:
      inputs, targets = batch["input"], batch["target"]
      logits = model(inputs)
      loss = loss_fn(logits, targets)

      if is_training:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

      batch_size = targets.size(0)
      total_loss += loss.item() * batch_size
      total_correct += (logits.argmax(dim=1) == targets).sum().item()
      total_samples += batch_size

  return total_loss / total_samples, 100.0 * total_correct / total_samples


def training_loop(model, optimizer, train_data_loader, validation_data_loader, run):
  loss_fn = nn.CrossEntropyLoss()

  for epoch in range(1, run.config.epochs + 1):
    train_loss, train_accuracy = run_epoch(model, train_data_loader, loss_fn, optimizer)
    validation_loss, validation_accuracy = run_epoch(model, validation_data_loader, loss_fn)

    run.log({
      "Epoch": epoch,
      "Training loss": train_loss,
      "Training accuracy (%)": train_accuracy,
      "Validation loss": validation_loss,
      "Validation accuracy (%)": validation_accuracy,
    })

    if epoch == 1 or epoch % 100 == 0 or epoch == run.config.epochs:
      print(
        f"Epoch {epoch}, "
        f"Training loss {train_loss:.4f}, accuracy {train_accuracy:.2f}%, "
        f"Validation loss {validation_loss:.4f}, accuracy {validation_accuracy:.2f}%"
      )

## 데이터 형식과 한 epoch 학습 검사

In [ ]:
class FakeRun:
  def __init__(self):
    self.config = SimpleNamespace(
      batch_size=2,
      epochs=1,
      learning_rate=1e-3,
      n_hidden_unit_list=[8, 8],
    )
    self.logged = []

  def log(self, metrics):
    self.logged.append(metrics)


class TitanicTrainingTest(unittest.TestCase):
  def test_data_loaders_use_titanic_classification_samples(self):
    train_loader, validation_loader = get_data(FakeRun())

    self.assertEqual(len(train_loader.dataset) + len(validation_loader.dataset), 891)
    batch = next(iter(train_loader))
    self.assertEqual(batch["input"].shape[1], 10)
    self.assertEqual(batch["target"].dtype, torch.int64)

  def test_one_epoch_logs_finite_loss_and_accuracy(self):
    run = FakeRun()
    samples = [
      {"input": torch.zeros(10), "target": torch.tensor(0)},
      {"input": torch.ones(10), "target": torch.tensor(1)},
      {"input": torch.full((10,), 2.0), "target": torch.tensor(0)},
    ]
    loader = DataLoader(samples, batch_size=2)
    model, optimizer = get_model_and_optimizer(run, n_input=10)

    self.assertEqual(model(torch.zeros(2, 10)).shape, (2, 2))
    training_loop(model, optimizer, loader, loader, run)

    self.assertEqual(len(run.logged), 1)
    metrics = run.logged[0]
    self.assertEqual(metrics["Epoch"], 1)
    for key in ("Training loss", "Validation loss"):
      self.assertTrue(math.isfinite(metrics[key]))
    for key in ("Training accuracy (%)", "Validation accuracy (%)"):
      self.assertGreaterEqual(metrics[key], 0)
      self.assertLessEqual(metrics[key], 100)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TitanicTrainingTest)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "학습 코드 검사 실패"

## 학습 설정

`USE_WANDB = True`로 바꾸면 W&B에 지표를 기록함.

In [ ]:
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
HIDDEN_UNITS = [20, 20]
USE_WANDB = False

## 모델 학습

In [ ]:
torch.manual_seed(42)
config = {
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "n_hidden_unit_list": HIDDEN_UNITS,
}

with wandb.init(
    mode="online" if USE_WANDB else "disabled",
    project="titanic_survival",
    tags=["titanic", "classification"],
    name=datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M-%S"),
    config=config,
) as run:
    run.define_metric("Training loss", step_metric="Epoch")
    run.define_metric("Validation loss", step_metric="Epoch", summary="min")
    run.define_metric("Training accuracy (%)", step_metric="Epoch")
    run.define_metric("Validation accuracy (%)", step_metric="Epoch", summary="max")

    train_data_loader, validation_data_loader = get_data(run)
    n_input = train_data_loader.dataset[0]["input"].numel()
    model, optimizer = get_model_and_optimizer(run, n_input)
    training_loop(model, optimizer, train_data_loader, validation_data_loader, run)